# PYTORCH MASTERY — NOTEBOOK 1
## Chapters 1–5: The PyTorch Ecosystem → Installation & Environment → Tensor Mastery → Tensor Data Types → Tensor Creation

**Prerequisite assumption:** You already know Python, NumPy, Pandas, Matplotlib/Seaborn, and Scikit-learn (classical ML, cross-validation, Random Forest, pipelines) at a working level. This course does not re-teach those — it builds deep learning engineering skill directly on top of them.


### PYTORCH MASTERY PROGRESS
Current Chapter: 5 | Current Level: Foundations

- [x] PyTorch Foundations
- [x] Tensor Mastery (started)
- [ ] Tensor Operations
- [ ] Autograd
- [ ] Neural Network Fundamentals
- [ ] *(...101 more chapters ahead)*


---
# CHAPTER 1 — THE PYTORCH ECOSYSTEM

## 1. Concept

```
NumPy (arrays, CPU-only, no gradients)
   ↓
PyTorch Tensor (arrays + GPU + automatic differentiation)
   ↓
Autograd (automatic gradient computation via a computational graph)
   ↓
nn.Module (composable, trainable neural network building blocks)
   ↓
Deep Learning (CNNs, RNNs, Transformers, LLMs, ...)
```

PyTorch is, at its foundation, **NumPy plus two things**: (1) the ability to run tensor operations on a GPU, and (2) automatic differentiation — the ability to compute gradients of any output with respect to any input automatically, by recording the sequence of operations as they happen. Everything else in PyTorch (`nn.Module`, optimizers, `DataLoader`) is built as convenient, composable infrastructure *on top of* those two core capabilities.

## 2. Why PyTorch Became Popular

PyTorch (Meta/Facebook AI Research, 2016) won broad adoption specifically because of its **define-by-run** (eager, dynamic) execution model: the computational graph is built fresh, on the fly, every time you run your Python code — as opposed to early TensorFlow's **define-then-run** (static graph) model, where you first fully specified a computation graph as a separate step, then executed it. Dynamic graphs mean:
- You can use ordinary Python control flow (`if`, `for`, recursion) directly inside your model's forward pass, and the graph adapts automatically.
- Debugging is direct — a normal Python debugger/`print()` statement works exactly as you'd expect, because your code genuinely executes line by line, not as a pre-compiled graph description.
- Research iteration is faster, which is why PyTorch became the dominant framework in academic research, and from there flowed into industry as those researchers became engineers.

## 3. PyTorch vs. NumPy

| | NumPy | PyTorch |
|---|---|---|
| Core object | `ndarray` | `Tensor` |
| Device | CPU only | CPU or GPU (`.to(device)`) |
| Gradients | None | Automatic differentiation via `autograd` |
| API similarity | — | Deliberately NumPy-like (`torch.zeros`, `torch.sum`, broadcasting rules) — this is intentional, to minimize your learning curve |
| Typical role | General numerical computing | Specifically deep learning (though usable for any GPU-accelerated numerical work) |

## 4. PyTorch vs. TensorFlow vs. JAX (Briefly, for Context)
- **TensorFlow** (Google) — historically static-graph (define-then-run); modern TensorFlow (2.x, via Keras and eager execution) has converged much closer to PyTorch's dynamic style. Still dominant in some large-scale production/mobile (TensorFlow Lite) contexts.
- **JAX** (Google) — a functional-programming-oriented array library built around composable function transformations (`grad`, `vmap`, `jit`); popular in research, especially where its `jit`-compiled, purely-functional style suits large-scale distributed training. Less "batteries-included" for typical deep learning workflows than PyTorch.
- **PyTorch** — currently the dominant framework in both research (the vast majority of new papers ship PyTorch code) and, increasingly, production, especially since PyTorch 2.x's `torch.compile` (Chapter 73) closed much of the historical "eager is slower than a compiled graph" performance gap.

## 5. PyTorch's Architecture — the Ecosystem, Layer by Layer


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available on this machine:", torch.cuda.is_available())
print("Number of CUDA devices:", torch.cuda.device_count())


PyTorch version: 2.11.0+cpu
CUDA available on this machine: False
Number of CUDA devices: 0


### Line-by-Line Explanation
- `torch.__version__` — always check this first when reading unfamiliar code or debugging: PyTorch's API has evolved meaningfully across major versions (2.x introduced `torch.compile`, changed some default behaviors), and "it works differently than the tutorial I read" is very often a version mismatch, not a misunderstanding.
- `torch.cuda.is_available()` — returns `True` only if PyTorch was installed with CUDA support **and** an NVIDIA GPU with a compatible driver is actually present. Returning `False` is extremely common (most laptops, this sandboxed environment, many cloud CPU-only instances) and is not itself an error — PyTorch runs entirely correctly on CPU, just slower for large models.
- `torch.cuda.device_count()` — `0` if no GPU is available; on a multi-GPU machine, this tells you how many devices exist for later chapters' multi-GPU discussion (Chapters 78–79).

## 6. The Core Modules You Will Use Constantly
| Module | Purpose |
|---|---|
| `torch` (top-level) | Tensor creation, math operations, `torch.save`/`load`, `torch.no_grad`, `torch.compile` |
| `torch.nn` | Neural network building blocks: layers, loss functions, containers (`nn.Module`, `nn.Sequential`) |
| `torch.optim` | Optimizers: SGD, Adam, AdamW, learning-rate schedulers |
| `torch.utils.data` | `Dataset`, `DataLoader` — efficient, parallelized data loading and batching |
| `torch.autograd` | The automatic differentiation engine (usually used implicitly via `.backward()`, rarely called directly) |
| `torch.cuda` | GPU device management and utilities |

## 7. What PyTorch Is NOT Primarily For
- **Classical ML** (Random Forest, Gradient Boosting on small-to-medium tabular data) — Scikit-learn remains the better tool there (Chapter 92 makes this comparison in full depth); PyTorch's overhead (building a full neural network, GPU management) is usually unjustified for problems a tree ensemble solves just as well or better.
- **Data wrangling** — that remains Pandas' job; PyTorch expects numeric tensors, not messy raw tables.

### Common Mistakes
- Assuming PyTorch will automatically use a GPU just because one exists — every tensor and model must be **explicitly** moved to the GPU device (Chapter 27); PyTorch defaults to CPU unless you tell it otherwise.
- Treating PyTorch and Scikit-learn as competitors rather than complementary tools solving different classes of problems — the honest, professional answer to "which do I use" is almost always "it depends on the data modality and problem," not "PyTorch is strictly better/newer."

### Interview Insight
**Q: What was PyTorch's key architectural differentiator that drove its adoption in research, and how has that gap with TensorFlow evolved?**
A: PyTorch's dynamic ("define-by-run") computational graph, built fresh on every forward pass using ordinary Python control flow, versus early TensorFlow's static ("define-then-run") graph that had to be fully specified before execution. This made PyTorch dramatically easier to debug and iterate with, driving its dominance in research. TensorFlow 2.x's eager execution mode closed much of this gap, and PyTorch 2.x's `torch.compile` closed much of the reverse performance gap (static graphs' compiled-execution speed advantage) — the frameworks have converged significantly, though PyTorch retains the dominant research/prototyping mindshare.

### Exercise
1. Run `import torch; print(torch.__version__)` in your own environment and record the version.
2. Check `torch.cuda.is_available()` — if `False`, note that every GPU-specific example in this course should still be read and understood conceptually; you can run the CPU path directly.

### Quiz (Chapter 1)
1. What are the two core capabilities PyTorch adds on top of what NumPy already provides?
2. What does "define-by-run" mean, and why did it matter for research adoption?
3. Name the four core `torch.*` submodules you'll use constantly, and each one's purpose.


---
# CHAPTER 2 — PYTORCH INSTALLATION AND ENVIRONMENT

## 1. Concept
Professional PyTorch setup means: an isolated environment, the *correct* PyTorch build for your hardware (CPU-only vs. a specific CUDA version), and a reproducible way for a teammate (or your future self) to recreate that exact environment.

## 2. Virtual Environments — Why They're Non-Negotiable
Different projects need different, sometimes conflicting, dependency versions (a legacy project pinned to `torch==1.13`, a new project needing `torch==2.x` for `torch.compile`). A virtual environment (`venv`, `conda`) isolates each project's dependencies from your system Python and from each other.

```bash
# venv (standard library, always available)
python -m venv pytorch-env
source pytorch-env/bin/activate     # Linux/Mac
pytorch-env\Scripts\activate        # Windows

# conda (popular in the ML community, handles non-Python deps like CUDA more gracefully)
conda create -n pytorch-env python=3.11
conda activate pytorch-env
```

## 3. Installing PyTorch — CPU vs. CUDA Builds

**Critical distinction**: `pip install torch` alone often installs a CPU-only build, or the wrong CUDA version for your GPU driver. The **correct, current** installation command should always be obtained from **pytorch.org's official install selector**, which generates the right command for your exact OS/package-manager/CUDA-version combination — this genuinely changes across PyTorch releases and CUDA versions, so don't trust a hardcoded command from an old tutorial (including, potentially, this one).

```bash
# Example shape of what pytorch.org's selector generates (illustrative — always verify current):
pip install torch --index-url https://download.pytorch.org/whl/cu121   # a CUDA 12.1 build
pip install torch                                                        # a CPU-only build
```

## 4. Verifying Your Installation


In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA version PyTorch was built with:", torch.version.cuda)
    print("GPU device name:", torch.cuda.get_device_name(0))
    print("Number of GPUs:", torch.cuda.device_count())
else:
    print("Running on CPU. Every tensor/model will default to the CPU device.")
    print("This is completely fine for learning and for small models -- just slower")
    print("for large models/datasets, where GPU acceleration (Chapter 27) matters a lot.")


PyTorch version: 2.11.0+cpu
CUDA available: False
Running on CPU. Every tensor/model will default to the CPU device.
This is completely fine for learning and for small models -- just slower
for large models/datasets, where GPU acceleration (Chapter 27) matters a lot.


### Diagnosing Common Installation Problems

| Symptom | Likely Cause | Fix |
|---|---|---|
| `torch.cuda.is_available()` returns `False` despite having a GPU | Installed the CPU-only build, or a CUDA build version-mismatched with your driver | Reinstall using pytorch.org's exact current selector command for your CUDA version |
| `RuntimeError: CUDA error: no kernel image is available` | PyTorch's CUDA build doesn't support your specific GPU's compute capability (common with very new or very old GPUs) | Check PyTorch's release notes for supported compute capabilities; may need a newer/older PyTorch version |
| `ImportError: DLL load failed` (Windows) | Missing Visual C++ Redistributable, or a corrupted install | Reinstall the VC++ Redistributable, then reinstall PyTorch |
| Works in one terminal, "module not found" in another | Multiple Python environments/interpreters; your IDE or terminal isn't using the environment where you installed PyTorch | Explicitly activate the correct virtual environment; check `which python` / `where python` |
| `CUDA out of memory` immediately, even on a small model | Another process (a previous, still-running Python process; another user on a shared GPU) is holding GPU memory | Check `nvidia-smi` (a command-line NVIDIA GPU monitoring tool) for other processes; restart the kernel |

### `nvidia-smi` — Your First GPU Diagnostic Tool
```bash
nvidia-smi
```
This command-line tool (installed alongside NVIDIA drivers, independent of PyTorch) shows: GPU utilization, memory usage, temperature, and every process currently using the GPU. **Always check this first** when debugging "CUDA out of memory" or "training is unexpectedly slow" — it tells you what's actually happening at the hardware level, information PyTorch's own error messages don't always surface clearly.

### Project Structure and `requirements.txt`
```
requirements.txt:
    torch==2.3.0
    torchvision==0.18.0
    numpy==1.26.4
```
Pinning **exact** versions (not just `torch>=2.0`) is standard practice for reproducibility — a later PyTorch release can subtly change default behaviors (Chapter 60's reproducibility lesson from the Scikit-learn course applies identically here).

### Common Mistakes
- Installing PyTorch via a generic `pip install torch` without checking whether it matches your actual GPU/CUDA setup — the single most common "why isn't my GPU being used" support question.
- Not using a virtual environment at all, leading to version conflicts between unrelated projects sharing one global Python installation.
- Forgetting that `torch.cuda.is_available()` being `False` doesn't mean something is broken — it may simply mean no GPU is present, which is a completely valid, common situation (this sandboxed course-writing environment included).

### Interview Insight
**Q: A colleague says "PyTorch isn't using my GPU" even though `nvidia-smi` shows the GPU is present and idle. What's your diagnostic process?**
A: First check `torch.cuda.is_available()` in Python — if `False`, the installed PyTorch build is likely CPU-only or mismatched with the installed CUDA driver version, requiring a reinstall via pytorch.org's current selector for the exact CUDA version `nvidia-smi` reports. If `True`, the issue is likely that the code itself never actually moves tensors/the model to the GPU device (`.to("cuda")`, Chapter 27) — PyTorch never uses the GPU automatically just because it's available; every tensor operation needs to explicitly happen on GPU-resident tensors.

### Quiz (Chapter 2)
1. Why should PyTorch's install command always come from pytorch.org's current selector rather than a hardcoded tutorial command?
2. What does `nvidia-smi` show that PyTorch's own Python API doesn't directly surface as clearly?
3. Does `torch.cuda.is_available()` returning `False` always indicate a broken installation?


---
# CHAPTER 3 — TENSOR MASTERY

## 1. Concept
A `torch.Tensor` is PyTorch's fundamental data structure — conceptually a NumPy `ndarray` with two additional superpowers: it can live on a GPU, and PyTorch can automatically track the operations performed on it to compute gradients later (Chapter 13). Every single piece of data in a PyTorch program — inputs, model weights, activations, gradients, outputs — is a tensor.

## 2. Tensor Creation — The Core Functions


In [ ]:
import torch

zeros = torch.zeros(3, 4)          # a 3x4 tensor of all zeros
ones = torch.ones(2, 3)            # a 2x3 tensor of all ones
empty = torch.empty(2, 2)          # UNINITIALIZED memory -- contains GARBAGE values, not zeros!
full = torch.full((2, 2), 7.0)     # a 2x2 tensor filled entirely with the value 7.0
arange = torch.arange(0, 10, 2)    # [0, 2, 4, 6, 8] -- like Python's range(), or np.arange()
linspace = torch.linspace(0, 1, 5) # 5 evenly spaced values from 0 to 1 INCLUSIVE
eye = torch.eye(3)                 # a 3x3 identity matrix
rand = torch.rand(2, 3)            # uniform random in [0, 1)
randn = torch.randn(2, 3)          # standard NORMAL random (mean=0, std=1)
randint = torch.randint(0, 10, (2, 3))  # random integers in [0, 10)

print("zeros:\n", zeros)
print("\nempty (GARBAGE values -- never assume this is zero-initialized!):\n", empty)
print("\narange:", arange)
print("linspace:", linspace)


zeros:
 tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])

empty (GARBAGE values -- never assume this is zero-initialized!):
 tensor([[1.1744e+32, 4.5539e-41],
        [1.1744e+32, 4.5539e-41]])

arange: tensor([0, 2, 4, 6, 8])
linspace: tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000])


### Line-by-Line Explanation
- `torch.zeros(3, 4)` — the arguments are the **shape** directly, not a shape *tuple* wrapped separately (though `torch.zeros((3, 4))` also works identically — PyTorch accepts both forms for shape-taking functions).
- `torch.empty(2, 2)` — this is a **critical, easy-to-misuse** function: it allocates memory without initializing it, so the tensor contains whatever garbage bytes happened to already be in that memory location. It exists purely for performance (skipping the initialization step) when you're about to immediately overwrite every value anyway — **never use `torch.empty()` expecting zeros or any particular values**.
- `torch.randn(2, 3)` vs `torch.rand(2, 3)` — a genuinely common naming confusion: `rand` = uniform `[0, 1)`, `randn` = standard **normal** (Gaussian) distribution. The "n" stands for "normal." This distinction matters enormously for weight initialization (Chapter 30), where the wrong choice can meaningfully affect training dynamics.

## 3. The Five Properties Every Tensor Has


In [ ]:
x = torch.randn(3, 4, requires_grad=True)

print("shape:", x.shape)              # torch.Size([3, 4]) -- the dimensions
print("ndim:", x.ndim)                # 2 -- the number of dimensions (the "rank")
print("dtype:", x.dtype)              # torch.float32 -- the default float type
print("device:", x.device)            # cpu (or cuda:0 if on GPU)
print("requires_grad:", x.requires_grad)  # True -- will this tensor's gradient be tracked?


shape: torch.Size([3, 4])
ndim: 2
dtype: torch.float32
device: cpu
requires_grad: True


### Line-by-Line Explanation
- `x.shape` — equivalent to `x.size()`; returns a `torch.Size` object (itself a subclass of Python's `tuple`), so it supports all normal tuple operations (`len(x.shape)`, indexing, unpacking).
- `x.ndim` — the number of dimensions, sometimes called the tensor's "rank" — a scalar has `ndim=0`, a vector `ndim=1`, a matrix `ndim=2`, a batch of images `ndim=4` (`batch, channels, height, width`).
- `x.dtype` — `torch.float32` is PyTorch's default for `torch.randn`/most creation functions; Chapter 4 covers the full dtype landscape and why it matters enormously for both numerical stability and memory/speed.
- `x.device` — `cpu` by default; a tensor and every tensor/model it interacts with in an operation **must be on the same device**, or PyTorch raises a `RuntimeError` (a very common early-career debugging encounter, revisited in Chapter 27).
- `x.requires_grad` — `True` here because we explicitly requested it; this is what makes autograd (Chapter 13) track every operation performed on `x` so gradients can later be computed with respect to it — essential for any tensor representing a *trainable parameter*, unnecessary (and wasteful) for tensors that are just data.

## 4. Tensor vs. NumPy Array — Head-to-Head


In [ ]:
import numpy as np

np_array = np.array([[1, 2, 3], [4, 5, 6]])
torch_tensor = torch.tensor([[1, 2, 3], [4, 5, 6]])

print("NumPy:", np_array.shape, np_array.dtype)
print("PyTorch:", torch_tensor.shape, torch_tensor.dtype)

# The APIs are DELIBERATELY similar -- this is intentional design, not coincidence
print("\nnp.sum(np_array, axis=0):", np.sum(np_array, axis=0))
print("torch.sum(torch_tensor, dim=0):", torch.sum(torch_tensor, dim=0))
print("\n(Note: NumPy calls it 'axis', PyTorch calls the SAME concept 'dim' -- a naming")
print(" difference to remember, not a functional one.)")


NumPy: (2, 3) int64
PyTorch: torch.Size([2, 3]) torch.int64

np.sum(np_array, axis=0): [5 7 9]
torch.sum(torch_tensor, dim=0): tensor([5, 7, 9])

(Note: NumPy calls it 'axis', PyTorch calls the SAME concept 'dim' -- a naming
 difference to remember, not a functional one.)


### When to Use NumPy vs. PyTorch Tensors
| Situation | Use |
|---|---|
| General numerical computing, no GPU need, no gradients needed | NumPy |
| Anything feeding into or coming out of a neural network | PyTorch tensor |
| Need automatic differentiation | PyTorch tensor (`requires_grad=True`) |
| Need GPU acceleration for large-scale numerical work, even non-DL | PyTorch tensor (GPU tensors work for general math too, not just neural nets) |
| Interfacing with Scikit-learn, Pandas, Matplotlib | NumPy (these libraries expect NumPy arrays; conversion is easy — Chapter 5, 93) |

### Common Mistakes
- Using `torch.empty()` and assuming it returns zeros — it does not; use `torch.zeros()` explicitly when you need zero-initialization.
- Confusing `axis` (NumPy's term) with `dim` (PyTorch's term) — they mean exactly the same thing, but using the wrong keyword name raises a `TypeError`.
- Setting `requires_grad=True` on every tensor "just in case" — wastes memory and computation tracking gradients for tensors (like raw input data) that never need them; reserve it for actual trainable parameters.
- Confusing `rand` (uniform) and `randn` (normal) — genuinely easy to typo given how similar the names look, and produces subtly different (not obviously broken) behavior.

### Interview Insight
**Q: What are the two capabilities that fundamentally distinguish a PyTorch Tensor from a NumPy ndarray?**
A: GPU support (a tensor can live and compute on a GPU via `.to("cuda")`, dramatically accelerating large-scale operations) and automatic differentiation (`requires_grad=True` lets PyTorch's autograd engine track operations on a tensor and later compute gradients with respect to it via `.backward()`). Everything else about the two objects' day-to-day API is deliberately near-identical by design, to make transitioning between them (and reasoning about PyTorch code if you already know NumPy) as easy as possible.

### Exercise
1. Create a `(4, 5)` tensor of all zeros, and a `(4, 5)` tensor of standard-normal random values. Print both tensors' `.shape`, `.dtype`, and `.device`.
2. Create a tensor with `requires_grad=True` and confirm `.requires_grad` reports correctly.

### Challenge
Write a function `describe_tensor(t)` that prints a tensor's shape, ndim, dtype, device, requires_grad, and — if the tensor has more than 100 elements — only a truncated preview (first 5 values) instead of the full tensor, to avoid flooding output on large tensors.

### Quiz (Chapter 3)
1. What are the five core properties every tensor has, and what does each tell you?
2. Why is `torch.empty()` dangerous if misused, and when is it actually the right choice?
3. What's the difference between `torch.rand()` and `torch.randn()`?
4. Why does PyTorch's API deliberately mirror NumPy's so closely?


---
# CHAPTER 4 — TENSOR DATA TYPES

## 1. Concept
A tensor's `dtype` determines how each element is stored in memory (how many bits, integer vs. floating point) — this affects **precision** (how exactly a value is represented), **memory** (bytes per element), and **speed** (how fast hardware can operate on that representation). Choosing the right dtype is a genuine engineering decision, not a formality.

## 2. The Floating-Point Family
| dtype | Bits | Typical use |
|---|---|---|
| `torch.float32` (`torch.float`) | 32 | **The default** — standard training precision, the safest general-purpose choice |
| `torch.float64` (`torch.double`) | 64 | Rarely used in deep learning (2x the memory, minimal accuracy benefit for typical DL workloads); occasionally used for numerically-sensitive scientific computing |
| `torch.float16` (`torch.half`) | 16 | Mixed-precision training/inference (Chapter 29) — half the memory of float32, faster on modern GPUs, but a much smaller representable range, risking numerical overflow/underflow |
| `torch.bfloat16` | 16 | "Brain float" — same exponent range as float32 (so much less overflow-prone than float16) but less precision (fewer mantissa bits); increasingly the preferred 16-bit format for training, especially for large models/LLMs |

## 3. The Integer and Boolean Family
| dtype | Bits | Typical use |
|---|---|---|
| `torch.int64` (`torch.long`) | 64 | **The required dtype for classification targets/labels** fed into `nn.CrossEntropyLoss` — a very common source of dtype errors (Chapter 33) |
| `torch.int32` | 32 | General integer indexing/counting |
| `torch.int8`, `torch.uint8` | 8 | Quantized model weights (Chapter 59), raw image pixel data (0–255) |
| `torch.bool` | 1 (stored) | Masks, boolean indexing |


In [ ]:
import torch

x_float32 = torch.tensor([1.0, 2.0, 3.0])                       # dtype inferred as float32 (the default)
x_float16 = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float16)   # explicit half precision
x_int64 = torch.tensor([1, 2, 3])                                 # dtype inferred as int64 for whole numbers
x_bool = torch.tensor([True, False, True])

print("x_float32 dtype:", x_float32.dtype, " element size:", x_float32.element_size(), "bytes")
print("x_float16 dtype:", x_float16.dtype, " element size:", x_float16.element_size(), "bytes")
print("x_int64 dtype:", x_int64.dtype, " element size:", x_int64.element_size(), "bytes")

# Converting between dtypes
x_converted = x_float32.to(torch.float16)   # or equivalently: x_float32.half()
print("\nConverted float32 -> float16:", x_converted.dtype)

# Memory comparison at scale
big_float32 = torch.zeros(1000, 1000, dtype=torch.float32)
big_float16 = torch.zeros(1000, 1000, dtype=torch.float16)
print(f"\n1000x1000 float32 tensor: {big_float32.element_size() * big_float32.nelement() / 1e6:.2f} MB")
print(f"1000x1000 float16 tensor: {big_float16.element_size() * big_float16.nelement() / 1e6:.2f} MB")


x_float32 dtype: torch.float32  element size: 4 bytes
x_float16 dtype: torch.float16  element size: 2 bytes
x_int64 dtype: torch.int64  element size: 8 bytes

Converted float32 -> float16: torch.float16

1000x1000 float32 tensor: 4.00 MB
1000x1000 float16 tensor: 2.00 MB


### Line-by-Line Explanation
- `x_float32.element_size()` — bytes per element (4 for float32, 2 for float16/bfloat16, 8 for int64); combined with `.nelement()` (total element count), this gives exact memory footprint — essential when reasoning about whether a model/batch will fit in GPU memory (Chapter 75).
- `.to(torch.float16)` — the general-purpose dtype conversion method; shorthand aliases like `.half()` (→float16), `.float()` (→float32), `.long()` (→int64), `.bool()` (→bool) also exist and are extremely common in real code.
- The 1000×1000 memory comparison directly demonstrates float16's **2x memory reduction** over float32 — at the scale of an actual large model (billions of parameters), this difference is the entire reason mixed-precision training (Chapter 29) and quantization (Chapter 59) exist as techniques.

### float16 vs. bfloat16 — the Overflow/Precision Tradeoff, Concretely


In [ ]:
# float16's SMALL representable range is a genuine numerical hazard --
# large activation values common in deep networks can silently OVERFLOW to infinity
large_value = torch.tensor([70000.0])   # a plausible activation magnitude deep in a large network

print("As float32:", large_value.to(torch.float32).item())
print("As float16:", large_value.to(torch.float16).item(), " <- OVERFLOWED to inf! float16's max is ~65504")
print("As bfloat16:", large_value.to(torch.bfloat16).item(), " <- fine, bfloat16 has float32's exponent RANGE")


As float32: 70000.0
As float16: inf  <- OVERFLOWED to inf! float16's max is ~65504
As bfloat16: 70144.0  <- fine, bfloat16 has float32's exponent RANGE


### Internal Working
Both float16 and bfloat16 use 16 bits total, but split them differently: float16 uses 1 sign + 5 exponent + 10 mantissa bits (a *narrow* representable range but *higher* precision within it); bfloat16 uses 1 sign + 8 exponent + 7 mantissa bits (the **same exponent range as float32**, so it can represent the same huge/tiny magnitudes float32 can, but with less precision — fewer significant digits). This is exactly why bfloat16 is generally the safer, more popular 16-bit choice for training large models: overflow (silently becoming `inf` or `NaN`, a Chapter 33 training-failure mode) is a much more damaging, harder-to-diagnose problem than modestly reduced precision.

### Common Mistakes
- Feeding `float32` or `float64` targets into `nn.CrossEntropyLoss` — it specifically requires `int64` (`.long()`) integer class-index targets, and passing a float tensor raises a clear but sometimes confusing error (Chapter 19, 33).
- Using `float16` for training without mixed-precision infrastructure (`GradScaler`, Chapter 29) — raw float16 training alone (not the managed "mixed precision" workflow) is genuinely prone to the overflow/underflow issue demonstrated above.
- Not checking `.dtype` when debugging an unexpected numerical result (`NaN`, `inf`) — dtype-related overflow is one of the first things to rule out.
- Assuming `torch.float64` is "more accurate and therefore always better" — for deep learning specifically, float32 is virtually always sufficient, and float64's 2x memory/compute cost is rarely justified.

### Best Practices
- Default to `float32` for training unless you have a specific, deliberate reason (mixed precision, quantization) to use something else.
- Prefer `bfloat16` over raw `float16` for training large models when using reduced precision, specifically because of its wider exponent range.
- Always use `int64` (`.long()`) for classification targets.

### Interview Insight
**Q: Why is bfloat16 often preferred over float16 for training large neural networks, even though both use the same total number of bits?**
A: They allocate their 16 bits differently — float16 uses more bits for precision (mantissa) but fewer for exponent range, giving it a narrow representable magnitude range (roughly ±65,504) that large models' activations can genuinely exceed, silently overflowing to `inf`. bfloat16 sacrifices some precision (fewer mantissa bits) to keep float32's much wider exponent range, making it far more resistant to the overflow failure mode — a more damaging and harder-to-diagnose training failure than bfloat16's modestly reduced precision typically causes in practice.

### Exercise
1. Create a tensor of small numbers (e.g., `[0.0001, 0.00001]`) and observe what happens when converted to `float16` — does it underflow to zero?
2. Convert a batch of image-like data (values 0–255) to `torch.uint8` and check its memory footprint versus the same data as `float32`.

### Quiz (Chapter 4)
1. What's the required dtype for classification targets fed into `CrossEntropyLoss`, and what error occurs if you use the wrong one?
2. Why does float16 risk overflow that bfloat16 mostly avoids, despite both being 16-bit?
3. When would `float64` actually be the right choice, if ever, in a deep learning context?


---
# CHAPTER 5 — TENSOR CREATION: NUMPY INTEROP AND `_like` FUNCTIONS

## 1. Concept
Beyond the basic creation functions (Chapter 3), this chapter covers the specific techniques for converting between NumPy and PyTorch, and for creating a new tensor that matches an existing tensor's shape/dtype/device — both extremely common in real code.

## 2. `torch.from_numpy()` vs. `torch.tensor()` vs. `torch.as_tensor()` — the Critical Memory-Sharing Distinction


In [ ]:
import numpy as np
import torch

np_array = np.array([1.0, 2.0, 3.0])

# torch.tensor() ALWAYS COPIES -- the new tensor is fully independent of the original array
tensor_copy = torch.tensor(np_array)

# torch.from_numpy() SHARES MEMORY -- modifying one modifies the other!
tensor_shared = torch.from_numpy(np_array)

np_array[0] = 999.0   # modify the ORIGINAL numpy array

print("torch.tensor() copy -- UNCHANGED:", tensor_copy)
print("torch.from_numpy() shared -- CHANGED too:", tensor_shared)


torch.tensor() copy -- UNCHANGED: tensor([1., 2., 3.], dtype=torch.float64)
torch.from_numpy() shared -- CHANGED too: tensor([999.,   2.,   3.], dtype=torch.float64)


### Line-by-Line Explanation
- `torch.tensor(np_array)` — always performs a full memory copy; the resulting tensor is completely independent — safe, but costs memory/time for large arrays.
- `torch.from_numpy(np_array)` — creates a tensor that **shares the exact same underlying memory buffer** as the NumPy array — genuinely fast (zero-copy) but means changes to either object are immediately visible in the other. This is a **frequent, subtle bug source**: modifying a NumPy array you thought was "safely converted" can silently corrupt a tensor already in use elsewhere (or vice versa).
- `torch.as_tensor(np_array)` — a middle-ground: it shares memory **when possible** (same as `from_numpy` for a compatible NumPy array) but will copy if a copy is genuinely required (e.g., converting from a Python list, or a NumPy array with a dtype/layout PyTorch can't directly share). It's the generally-recommended, safer default when you don't specifically need `tensor()`'s guaranteed-copy behavior or `from_numpy()`'s guaranteed-share behavior.

## 3. Going the Other Direction: Tensor → NumPy


In [ ]:
t = torch.tensor([1.0, 2.0, 3.0])

# .numpy() shares memory with the tensor (same sharing behavior as from_numpy(), just reversed)
back_to_numpy = t.numpy()
t[0] = 555.0
print("Modified tensor:", t)
print("NumPy view also changed (shared memory):", back_to_numpy)

# CRITICAL: .numpy() FAILS on a GPU tensor OR a tensor with requires_grad=True still attached to a graph
t_grad = torch.tensor([1.0, 2.0], requires_grad=True)
try:
    t_grad.numpy()
except RuntimeError as e:
    print("\nError on a requires_grad tensor:", str(e)[:150])

# The fix: .detach() first (removes it from the computational graph), THEN .numpy()
safe_numpy = t_grad.detach().numpy()
print("\nAfter .detach():", safe_numpy)


Modified tensor: tensor([555.,   2.,   3.])
NumPy view also changed (shared memory): [555.   2.   3.]

Error on a requires_grad tensor: Can't call numpy() on Tensor that requires grad. Use tensor.detach().numpy() instead.

After .detach(): [1. 2.]


### Line-by-Line Explanation
- `.numpy()` raises `RuntimeError` on a tensor with `requires_grad=True` — because NumPy has no concept of a computational graph, and silently converting would either lose that tracking information or create an inconsistent state; PyTorch forces you to be explicit about discarding gradient tracking first.
- `.detach()` — returns a new tensor that shares the same data but is deliberately **disconnected** from the autograd computational graph (fully explained in Chapter 15) — the correct, explicit way to say "I'm done tracking gradients for this value, give me a plain tensor."
- For a **GPU** tensor, `.numpy()` also fails directly (NumPy is CPU-only) — the fix there is `.cpu().numpy()` (move to CPU first, then convert), or combined: `.detach().cpu().numpy()` for a tensor that's both on GPU and part of a gradient-tracked computation — a genuinely common, near-idiomatic line you'll write constantly when extracting final results for plotting/analysis (Chapter 94).

## 4. The `_like` Family — Creating Tensors Matching an Existing One's Shape/dtype/Device


In [ ]:
reference = torch.randn(3, 4, dtype=torch.float64, device="cpu")

zeros_like_ref = torch.zeros_like(reference)
ones_like_ref = torch.ones_like(reference)
rand_like_ref = torch.rand_like(reference)
randn_like_ref = torch.randn_like(reference)

print("reference dtype/device:", reference.dtype, reference.device)
print("zeros_like dtype/device (MATCHES reference):", zeros_like_ref.dtype, zeros_like_ref.device)
print("zeros_like shape (MATCHES reference):", zeros_like_ref.shape)


reference dtype/device: torch.float64 cpu
zeros_like dtype/device (MATCHES reference): torch.float64 cpu
zeros_like shape (MATCHES reference): torch.Size([3, 4])


### Why the `_like` Functions Matter in Real Code
These are genuinely common in model code — e.g., initializing a hidden state tensor that must exactly match an input batch's shape, dtype, and device without hardcoding any of those three properties (which would break the moment the model runs on GPU instead of CPU, or with a different batch size or precision). Writing `torch.zeros(batch_size, hidden_dim, device=device, dtype=dtype)` manually is more error-prone and verbose than `torch.zeros_like(some_reference_tensor)` when a suitable reference tensor is already available.

### Common Mistakes
- Using `torch.from_numpy()` and then being surprised when modifying the original NumPy array (e.g., in a later Pandas/NumPy processing step) silently changes an already-in-use tensor.
- Calling `.numpy()` directly on a GPU tensor or a `requires_grad=True` tensor without `.cpu()`/`.detach()` first — a very frequent early-PyTorch error, precisely because the fix (`.detach().cpu().numpy()`) needs to be memorized as an idiom.
- Hardcoding a tensor's device/dtype when creating a "matching" tensor manually, instead of using the appropriate `_like` function — brittle to changes in the reference tensor's actual device/dtype.

### Best Practices
- Use `torch.as_tensor()` as your default NumPy→tensor conversion unless you specifically need `tensor()`'s guaranteed copy or `from_numpy()`'s guaranteed share.
- Standardize on `.detach().cpu().numpy()` as the idiom for "get me a plain NumPy array from this tensor, wherever it happens to live and however it's tracked" — it's always correct, even when `.detach()` or `.cpu()` individually would be unnecessary for a specific tensor (a `.cpu()` call on an already-CPU tensor is a harmless no-op).

### Interview Insight
**Q: What's the difference between `torch.tensor()` and `torch.from_numpy()`, and why does this distinction matter in practice?**
A: `torch.tensor()` always performs a full memory copy, producing a tensor completely independent of the source data; `torch.from_numpy()` shares the underlying memory buffer with the original NumPy array — modifying one is immediately visible in the other. This matters because `from_numpy()`'s zero-copy behavior is faster and more memory-efficient for large arrays, but creates a subtle correctness risk: any later modification to the original NumPy array (e.g., in a separate data-processing step) silently mutates the tensor too, which can produce very confusing bugs if the sharing relationship isn't kept in mind. `torch.as_tensor()` is generally the safer default, sharing memory when it safely can while still copying when necessary.

### Exercise
1. Convert a Pandas DataFrame column to a NumPy array, then to a PyTorch tensor using `torch.as_tensor()`. Verify the resulting tensor's dtype.
2. Create a `requires_grad=True` tensor on CPU, attempt `.numpy()` directly (observe the error), then correctly extract a NumPy array from it.

### Challenge
Write a function `safe_to_numpy(t)` that correctly converts ANY tensor — regardless of device, dtype, or `requires_grad` status — to a NumPy array, handling every edge case from this chapter in one robust utility function.

### Quiz (Chapter 5)
1. Which of `torch.tensor()`, `torch.from_numpy()`, `torch.as_tensor()` guarantees a copy, which guarantees sharing, and which does either depending on the situation?
2. Why does calling `.numpy()` on a `requires_grad=True` tensor raise an error, and what's the fix?
3. Why would you use `torch.zeros_like(reference)` instead of manually specifying shape/dtype/device?

### Chapter 5 Summary
- `torch.tensor()` copies; `torch.from_numpy()` shares memory; `torch.as_tensor()` shares when possible.
- Converting a tensor to NumPy requires `.detach().cpu().numpy()` for any tensor that might be gradient-tracked and/or GPU-resident — memorize this idiom.
- The `_like` family creates new tensors matching an existing tensor's shape/dtype/device — safer and more maintainable than hardcoding those properties.

---

## END OF NOTEBOOK 1 — What You Should Now Be Able To Do
- Explain what PyTorch adds on top of NumPy (GPU + autograd) and where it fits relative to Scikit-learn.
- Diagnose common installation/CUDA-availability problems and know `nvidia-smi` as your first GPU diagnostic tool.
- Create tensors using every core creation function, and correctly read every tensor's five defining properties (shape, ndim, dtype, device, requires_grad).
- Choose the correct dtype deliberately (float32 default, bfloat16 over float16 for reduced-precision training, int64 for classification targets) and explain the float16 overflow risk concretely.
- Convert fluently and correctly between NumPy and PyTorch in both directions, including the `.detach().cpu().numpy()` idiom for gradient-tracked/GPU tensors.

## Before Moving to Notebook 2 (Chapters 6–10: Indexing and Slicing, Reshaping, Broadcasting, Mathematical Operations, Matrix Operations)

